In [ ]:
#Installing Dependencies
!pip install tensorflow scikit-learn pandas numpy


In [ ]:
!pip install pandas numpy scikit-learn scipy m2cgen


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 kB 6.4 MB/s eta 0:00:00


In [ ]:
# Import Libraries
import numpy as np
import pandas as pd

from scipy.stats import mode
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import m2cgen as m2c



In [ ]:
from google.colab import files
uploaded = files.upload()


Saving CompleteDataSet.csv to CompleteDataSet (2).csv


In [ ]:
import os
print(os.listdir())


['.config', 'CompleteDataSet.csv', 'upfall_wrist_model.h5', 'upfall_wrist_model.tflite', 'CompleteDataSet (1).csv', 'CompleteDataSet (2).csv', 'upfall_wrist_model_float32.tflite', 'upfall_wrist_model_int8.cc', 'upfall_wrist_model_int8.tflite', 'sample_data']


In [ ]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
WINDOW_SIZE = 50   # ~2.5 seconds of data
STEP_SIZE   = 12   # 75% overlap

# Wrist IMU columns
SENSOR_COLS = [
    "device6_acc_x", "device6_acc_y", "device6_acc_z",
    "device6_gyro_x", "device6_gyro_y", "device6_gyro_z",
    "acc_mag",
    "gyro_mag"
]


# ==========================================
# 2. FEATURE EXTRACTION
# ==========================================
def extract_features_pico(window):
    """
    Calculates:
      - Mean, Std, Min, Max, Range, Energy, Jerk
    for each of the 8 signals:
      3x acc, 3x gyro, acc_mag, gyro_mag
    => 8 * 7 = 56 features per window.
    """
    features = []
    num_signals = window.shape[1]  # should be 8 now

    for i in range(num_signals):
        axis_data = window[:, i]

        mean_val = np.mean(axis_data)
        std_val  = np.std(axis_data)
        max_val  = np.max(axis_data)
        min_val  = np.min(axis_data)
        range_val = max_val - min_val

        energy = np.sum(axis_data ** 2) / len(axis_data)

        diff = np.diff(axis_data)
        jerk = np.sum(np.abs(diff))

        features.extend([
            mean_val,
            std_val,
            max_val,
            min_val,
            range_val,
            energy,
            jerk
        ])

    return np.array(features)




In [ ]:
# ==========================================
# 3. LOAD & MAP DATA
# ==========================================
print("Loading dataset...")
df = pd.read_csv("CompleteDataSet.csv")


new_header = df.iloc[0]
df = df[1:]
df.columns = new_header
df = df.reset_index(drop=True)

# Build column names (time + 6 devices * 7 signals + metadata)
columns = ["TimeStamps"]
imu_names = ["acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z","lux"]

for i in range(1, 7):
    for name in imu_names:
        columns.append(f"device{i}_{name}")

columns += ["Subject", "Activity", "Trial", "Tag"]
df.columns = columns


def map_5_classes(tag):
    try:
        t = int(float(tag))
    except:
        return 2


    if 1 <= t <= 5:
        return 0


    elif t in [6, 10]:
        return 1


    elif t in [7, 9]:
        return 2


    elif t == 8:
        return 3


    elif t == 11:
        return 4


    return 2

df["label_class"] = df["Tag"].apply(map_5_classes)


df["acc_mag"] = np.sqrt(
    df["device6_acc_x"].astype(float)**2 +
    df["device6_acc_y"].astype(float)**2 +
    df["device6_acc_z"].astype(float)**2
)

df["gyro_mag"] = np.sqrt(
    df["device6_gyro_x"].astype(float)**2 +
    df["device6_gyro_y"].astype(float)**2 +
    df["device6_gyro_z"].astype(float)**2
)


X_raw = df[SENSOR_COLS].astype(float).values
y_raw = df["label_class"].values




print("Dataset mapped to 5 Classes:")
print("0=Fall, 1=Active, 2=Stand, 3=Sit, 4=Lay")
print(df["label_class"].value_counts().sort_index())


Loading dataset...


/tmp/ipython-input-3475977290.py:5: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("CompleteDataSet.csv")


Dataset mapped to 5 Classes:
0=Fall, 1=Active, 2=Stand, 3=Sit, 4=Lay
label_class
0     8516
1    81406
2    68573
3    54027
4    82156
Name: count, dtype: int64


In [ ]:
df.head()


,TimeStamps,device1_acc_x,device1_acc_y,device1_acc_z,device1_gyro_x,device1_gyro_y,device1_gyro_z,device1_lux,device2_acc_x,device2_acc_y,...,device6_gyro_y,device6_gyro_z,device6_lux,Subject,Activity,Trial,Tag,label_class,acc_mag,gyro_mag
0,2018-07-04T12:04:17.738369,-1.005,0.229,-0.083,-0.671,0.488,-2.683,0,-0.981,0.26,...,1.0,1.0,1.0,1.0,1.0,1.0,7.0,2,45.022217,1.732051
1,2018-07-04T12:04:17.790509,-1.005,0.228,-0.082,-3.415,-0.549,0.122,0,-0.981,0.26,...,1.0,1.0,1.0,1.0,1.0,1.0,7.0,2,1.732051,1.732051
2,2018-07-04T12:04:17.836632,-1.005,0.231,-0.079,-2.622,-1.402,-0.549,0,-0.975,0.282,...,1.0,1.0,1.0,1.0,1.0,1.0,7.0,2,325.003077,1.732051
3,2018-07-04T12:04:17.885262,-1.005,0.231,-0.079,-2.561,-2.195,-1.22,0,-0.973,0.301,...,1.0,1.0,1.0,1.0,1.0,1.0,7.0,2,396.002525,1.732051
4,2018-07-04T12:04:17.945423,-1.008,0.229,-0.072,-3.537,-2.073,-0.305,0,-0.973,0.301,...,1.0,1.0,1.0,1.0,1.0,1.0,7.0,2,436.002294,1.732051


In [ ]:
# ==========================================
# 4. WINDOWING
# ==========================================
print("Creating windows...")

X_features = []
y_labels = []

for start in range(0, len(X_raw) - WINDOW_SIZE, STEP_SIZE):
    end = start + WINDOW_SIZE
    window = X_raw[start:end]
    labels = y_raw[start:end]


    if 0 in labels:
        label = 0
    else:

        label = mode(labels, keepdims=True)[0][0]

    feats = extract_features_pico(window)
    X_features.append(feats)
    y_labels.append(label)

X_features = np.array(X_features)
y_labels = np.array(y_labels)

print("Feature matrix shape:", X_features.shape)
print("Window labels shape:", y_labels.shape)
print("Class distribution in windows:", dict(zip(*np.unique(y_labels, return_counts=True))))



Creating windows...
Feature matrix shape: (24553, 56)
Window labels shape: (24553,)
Class distribution in windows: {np.int64(0): np.int64(1758), np.int64(1): np.int64(6774), np.int64(2): np.int64(5360), np.int64(3): np.int64(4487), np.int64(4): np.int64(6174)}


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


unique, counts = np.unique(y_labels, return_counts=True)
print("Original window counts per class:", dict(zip(unique, counts)))

X_final = X_features
y_final = y_labels


fall_class = 0
idx_fall = np.where(y_labels == fall_class)[0]
idx_nonfall = np.where(y_labels != fall_class)[0]

X_fall = X_features[idx_fall]
y_fall = y_labels[idx_fall]
X_nonfall = X_features[idx_nonfall]
y_nonfall = y_labels[idx_nonfall]

print("Original #Fall windows:", len(X_fall))
print("Original #Non-Fall windows:", len(X_nonfall))


target_fall = min(len(X_nonfall) // 4, len(X_fall) * 10)

if target_fall > len(X_fall):
    from sklearn.utils import resample

    X_fall_up, y_fall_up = resample(
        X_fall, y_fall,
        replace=True,
        n_samples=target_fall,
        random_state=42
    )

    X_bal = np.concatenate([X_nonfall, X_fall_up], axis=0)
    y_bal = np.concatenate([y_nonfall, y_fall_up], axis=0)
else:
    X_bal = X_final
    y_bal = y_final

print("Balanced dataset shape:", X_bal.shape)

# Train-test split on balanced set
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal,
    test_size=0.3,
    stratify=y_bal,
    random_state=42
)


rf = RandomForestClassifier(
    n_estimators=80,          # more trees than 60
    max_depth=12,            # a bit deeper than 10
    min_samples_leaf=5,      # regularization
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest...")
rf.fit(X_train, y_train)




Original window counts per class: {np.int64(0): np.int64(1758), np.int64(1): np.int64(6774), np.int64(2): np.int64(5360), np.int64(3): np.int64(4487), np.int64(4): np.int64(6174)}
Original #Fall windows: 1758
Original #Non-Fall windows: 22795
Balanced dataset shape: (28493, 56)
Training Random Forest...


RandomForestClassifier(class_weight='balanced_subsample', max_depth=12,
                       min_samples_leaf=5, n_estimators=80, n_jobs=-1,
                       random_state=42)

In [ ]:


def map_to_5_classes(tag):
    tag = int(tag)


    if tag in [1, 2, 3, 4, 5]:
        return 0


    elif tag == 6:
        return 1
    elif tag == 7:
        return 2
    elif tag == 8:
        return 3
    else:

        return 4   # Other ADL (Jumping, Laying, etc.)


y_5class = np.array([map_to_5_classes(t) for t in y_windows])

print("Original Tag labels:", np.unique(y_windows))
print("New 5-class labels:", np.unique(y_5class))



Original Tag labels: [ 1  2  3  4  5  6  7  8  9 10 11 20]
New 5-class labels: [0 1 2 3 4]


In [ ]:
# ==========================================
# 6. EVALUATION
# ==========================================
target_names = ["FALL (0)", "ACTIVE (1)", "STAND (2)", "SIT (3)", "LAY (4)"]

y_pred = rf.predict(X_test)

print("\n--- 5-CLASS MODEL RESULTS ---")
print(classification_report(y_test, y_pred, target_names=target_names))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))




--- 5-CLASS MODEL RESULTS ---
              precision    recall  f1-score   support

    FALL (0)       0.79      0.89      0.83      1710
  ACTIVE (1)       0.90      0.84      0.87      2032
   STAND (2)       0.57      0.70      0.63      1608
     SIT (3)       0.63      0.44      0.52      1346
     LAY (4)       0.82      0.78      0.80      1852

    accuracy                           0.75      8548
   macro avg       0.74      0.73      0.73      8548
weighted avg       0.75      0.75      0.75      8548

Confusion Matrix:
[[1516   63   33   14   84]
 [ 184 1710   64   40   34]
 [  50   42 1133  236  147]
 [  24   43  625  592   62]
 [ 149   43  144   63 1453]]


In [ ]:
# ==========================================
# 7. EXPORT MODEL TO C (for Pico)
# ==========================================
print("\nGenerating C code with m2cgen...")

code = m2c.export_to_c(rf)

with open("pico_5class_model.h", "w") as f:
    f.write(code)

print("Saved to 'pico_5class_model.h'")



Generating C code with m2cgen...
Saved to 'pico_5class_model.h'


In [ ]:
from google.colab import files
files.download("pico_5class_model.h")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>